## creating sentence embeddings for confluence pages

Convert sentences from Confluence pages into embeddings using the saved model from Hugging Face (which we fine-tuned in the fine_tune_hugging_face_model.ipynb notebook).

In [ ]:
import sentence_transformers as st
import pandas as pd
import os, sys

sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from confluence_api.getting_confluence_content import *

In [ ]:
# getting data from confluence pages
chunk_size = 5
no_pages = 5

pages = get_page_info(no_pages = no_pages, space_key = 'DA')
pages.columns = ['page_id', 'title', 'note']

# loading model
model = st.SentenceTransformer('model_2')

# loading notes with assigned vectors representing a meaning
documentation_embeddings_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'documentation_embeddings.json')
if not os.path.isfile(documentation_embeddings_path):
    doc_embed = pd.DataFrame(columns = ['note', 'embeddings'], data = [])
else:
    doc_embed = pd.read_json(documentation_embeddings_path)

In [ ]:
# dividing text into chunks
def divide_text(text, chunk_size):
    # chunk_size is a number of words per chunk
    chunks = []
    for i in range(
        0, 
        len(text.split()) - (len(text.split()) % chunk_size), 
        chunk_size // 2
    ):
        chunks.append(' '.join(text.split()[i : i + chunk_size]))

    chunks.append(' '.join(text.split()[len(text.split()) - chunk_size : ]))
    
    return chunks

In [ ]:
# creating sentence embedding for confluence page based on its id
def encode_page(page_id, model):
    page_content = get_text_content(page_id)
    # if there is a lot of text then divide it into smaller chunks
    if len(page_content) > chunk_size:
        chunks = divide_text(page_content, chunk_size)
        embeddings = [model.encode(chunk) for chunk in chunks]
    else:
        embeddings = [model.encode(page_content)]
        
    return embeddings

In [ ]:
# take only those rows from pages which are not already in doc_embed
pages = pages.merge(doc_embed, on = 'note', how = 'left')
pages = pages[pages.embeddings.isnull()]

In [ ]:
pages.shape

(5, 4)

In [ ]:
pages['embeddings'] = pages.page_id.apply(lambda x: encode_page(x, model))

In [ ]:
pages

,page_id,title,note,embeddings
0,124993322,Sisense On-boarding Information,https://confluence.sdl.com/display/DA/Sisense+...,"[[0.09389132, -0.44781134, -0.18386544, 0.3037..."
1,124993442,Sisense Webinars & Tutorials,https://confluence.sdl.com/pages/viewpage.acti...,"[[0.09389132, -0.44781134, -0.18386544, 0.3037..."
2,151633202,Reversing the Order of Dates in Pivot Table (n...,https://confluence.sdl.com/pages/viewpage.acti...,"[[-0.09924609, -0.67648005, -0.24289468, -0.05..."
3,153851192,Systems Admin Details,https://confluence.sdl.com/display/DA/Systems+...,"[[0.044606596, -0.37054223, -0.36413017, 0.122..."
4,158438583,Sort Column chart with Break-by values,https://confluence.sdl.com/display/DA/Sort+Col...,"[[-0.16207242, -0.4994395, -0.34958267, 0.1816..."


In [ ]:
doc_embed = pd.concat((doc_embed, pages[['note', 'embeddings']]))
doc_embed.reset_index(drop = True, inplace = True)

In [ ]:
doc_embed.to_json(documentation_embeddings_path)